# Z3 IR Verifier Regression Evaluation

Evaluate the revised verifier against synthetic constraints and saved pilot IRs, and check result aggregation and reference adjudication. All evaluation uses CPU computation without model API requests.

## 1. Environment and evaluation inputs

Install Z3 and upload the [repository archive at commit cfe5051](https://github.com/sam134-eng/Risk-aware-financial-llm/archive/cfe50516e291715038ffc7589844e4321e513273.zip). Verify both verifier sources, the two pilot inputs, and the adjudication CSV against their Git blob identifiers before execution.

In [ ]:
%pip -q install z3-solver==4.15.4.0

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import importlib.util
import json
import tempfile
import zipfile
from collections import Counter
import z3

COMMIT = "cfe50516e291715038ffc7589844e4321e513273"
BLOBS = {
    "scripts/verify_ir_with_z3 (Updated).py": "8015977361983e450af5e889380f20cdf41be417",
    "scripts/verify_ir_with_z3.py": "7cae16243af87ca952248172b62bfb879555d4fb",
    "pilot_3shot.jsonl": "81bf25830d933bc629ec2211fcd839f90034ce42",
    "pilot_8shot.jsonl": "fd5e633c3f6633c464250aafdef5d394fb8a3728",
    "results/gold_audit/gold_adjudication_30.csv": "b321ff4dc9e77e98c1614cc0046bea8afbdb7f6b",
}

def git_blob(raw):
    return hashlib.sha1(b"blob " + str(len(raw)).encode() + b"\0" + raw).hexdigest()

uploaded = files.upload()
assert len(uploaded) == 1, "Upload one repository ZIP."
archive = Path(next(iter(uploaded)))
work = Path(tempfile.mkdtemp(prefix="finverify_z3_"))
updated_path, baseline_path = list(BLOBS)[:2]
with zipfile.ZipFile(archive) as z:
    suffix = "/" + updated_path
    matches = [name for name in z.namelist() if name.endswith(suffix)]
    assert len(matches) == 1, "Updated verifier not found. Use the pinned repository ZIP."
    prefix = matches[0][:-len(suffix)]
    for rel, expected in BLOBS.items():
        raw = z.read(prefix + "/" + rel)
        assert git_blob(raw) == expected, f"Git blob mismatch: {rel}"
        dest = work / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(raw)
        print("Verified:", rel)

def load(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module

updated = load("updated_verifier", work / updated_path)
old = load("previous_verifier", work / baseline_path)
print("Z3 version:", z3.get_version_string())
print("Verifier imports: PASS")


## 2. Synthetic regression cases

Expected statuses follow the revised verifier's tolerance-based claim semantics. PASS means every feasible target lies within tolerance; FAIL means no feasible target lies within tolerance; UNDERDETERMINED means feasible targets exist both inside and outside tolerance. A nonunique target can therefore receive FAIL if the claim is impossible, or PASS if all feasible values fall within tolerance. These are not tests of exact target uniqueness. The fixed-target mismatch also checks the returned unsatisfiable core.

In [ ]:
def ir(eqs, claim, variables=None, tol=0.01):
    return {"variables": {"x": 10} if variables is None else variables,
            "equations": [{"lhs": lhs, "rhs": rhs} for lhs, rhs in eqs],
            "claimed_answer": {"target_variable": "y", "predicted_value": claim, "tolerance": tol}}

cases = [
    ("contradictory_matching_claim", ir([("y","x+5"),("y","x+10")],15), "INCONSISTENT_IR"),
    ("contradictory_nonmatching_claim", ir([("y","x+5"),("y","x+10")],100), "INCONSISTENT_IR"),
    ("unconstrained_target", ir([("y","y")],15), "UNDERDETERMINED"),
    ("nonlinear_multiple_roots_claim_possible", ir([("y","y*y")],1), "UNDERDETERMINED"),
    ("nonlinear_multiple_roots_claim_impossible", ir([("y","y*y")],3), "FAIL"),
    ("free_auxiliary_fixed_target_match", ir([("u","u"),("y","x+5")],15), "PASS"),
    ("free_auxiliary_fixed_target_mismatch", ir([("u","u"),("y","x+5")],20), "FAIL"),
    ("unique_nonlinear_target", ir([("y","y"),("z","y*y"),("z","0")],0), "PASS"),
    ("division_by_zero", ir([("y","x/0")],0), "INCONSISTENT_IR"),
    ("tolerance_boundary", ir([("y","10.01")],10), "PASS"),
    ("outside_tolerance", ir([("y","10.011")],10), "FAIL"),
    ("negative_tolerance", ir([("y","10")],10,tol=-0.01), "NOT_APPLICABLE"),
    ("lower_tolerance_boundary", ir([("y","9.99")],10), "PASS"),
    ("nonzero_divisor", ir([("y","x/2")],5), "PASS"),
]
results=[]
for name, obj, expected in cases:
    try:
        verdict=updated.verify_ir_entry(obj)
        actual=verdict["status"]
        results.append({"case":name,"expected":expected,"actual":actual,"match":actual==expected,"verification":verdict})
    except Exception as e:
        results.append({"case":name,"expected":expected,"actual":"EXCEPTION","match":False,"error":repr(e)})
    print(name, results[-1]["actual"], "OK" if results[-1]["match"] else "REVIEW")
print("Regression checks matched:",sum(r["match"] for r in results),"/",len(results))
by_name = {r["case"]: r for r in results}
fixed = by_name["free_auxiliary_fixed_target_mismatch"]
core = fixed.get("verification", {}).get("unsat_core", [])
# A core may vary across solvers; verify the necessary claim and defining constraints.
feedback = {
    "claim_in_core": "claimed_answer" in core,
    "defining_equation_in_core": "eq:2::y" in core,
    "input_binding_in_core": "var:x" in core,
    "core": core,
}
feedback["passed"] = all(feedback[k] for k in (
    "claim_in_core", "defining_equation_in_core", "input_binding_in_core"))
print("Unsat-core check:", feedback)


## 3. Paired pilot comparison

Compare both implementations on the same 16 records per configuration without reference overrides or exclusions. Report status changes and execution exceptions separately.

In [ ]:
report = {
    "repository_commit": COMMIT,
    "input_git_blobs": BLOBS,
    "z3_version": z3.get_version_string(),
    "regression": results,
    "unsat_core_feedback": feedback,
    "pilot_comparison": {},
}
for label in ("3shot","8shot"):
    rows=[json.loads(line) for line in (work/f"pilot_{label}.jsonl").read_text().splitlines() if line.strip()]
    assert len(rows) == 16, f"Unexpected {label} record count"
    comparisons=[]
    for i,row in enumerate(rows):
        cid=str(row.get("case_id",row.get("question_id",i+1)))
        obj=row.get("ir",row)
        try:
            a=old.verify_ir_entry(obj)
            b=updated.verify_ir_entry(obj)
            comparisons.append({"case_id":cid,"old":a["status"],"new":b["status"],"updated_verdict":b})
        except Exception as e:
            comparisons.append({"case_id":cid,"error":repr(e)})
    valid=[r for r in comparisons if "error" not in r]
    changes=[r for r in valid if r["old"]!=r["new"]]
    report["pilot_comparison"][label]=comparisons
    print(label, "records:",len(rows),"old:",dict(Counter(r["old"] for r in valid)),"updated:",dict(Counter(r["new"] for r in valid)))
    print("Changed:",[(r["case_id"],r["old"],r["new"]) for r in changes],"Exceptions:",len(rows)-len(valid))



## 4. JSONL aggregation

Check the per-record statuses, per-status counts and identifiers, and total record count against individual verifier calls. UNKNOWN is included in the aggregation schema but is not forced by the synthetic cases.

In [ ]:
input_file = work / "synthetic_cases.jsonl"
input_file.write_text("".join(json.dumps({"case_id": name, "ir": obj}) + "\n"
                              for name, obj, _ in cases), encoding="utf-8")
try:
    output_file = work / "synthetic_verdicts.json"
    updated.run_on_jsonl(input_file, output_file, use_adjudicated_refs=False,
                        exclude_adjudicated=False)
    out = json.loads(output_file.read_text())
    individual = {r["case"]: r["actual"] for r in results}
    aggregate = {r["case_id"]: r["verification"]["status"] for r in out["results"]}
    counts = Counter(individual.values())
    expected_counts = {s: counts[s] for s in updated.STATUSES}
    expected_ids = {s: sorted(k for k, v in individual.items() if v == s)
                    for s in updated.STATUSES}
    report["aggregation"] = {
        "matches_individual": individual == aggregate,
        "status_counts_match": out["summary"] == expected_counts,
        "status_ids_match": {k: sorted(v) for k, v in out["case_ids"].items()} == expected_ids,
        "record_count_matches": len(out["results"]) == len(cases),
        "total_count_matches": sum(out["summary"].values()) == len(cases),
        "output": out,
    }
except Exception as e:
    report["aggregation"] = {"error": repr(e)}
print("Aggregation:", {k: v for k, v in report["aggregation"].items() if k != "output"})


## 5. Reference adjudication

The agreed policy excludes six cases and supplies reference values for 24. A reference override evaluates compatibility with a reference value instead of the original model claim. These are distinct evaluation questions and are reported separately.

In [ ]:
csv_path = work / "results/gold_audit/gold_adjudication_30.csv"
try:
    excluded, refs = updated.load_adjudicated_overrides(csv_path)
    example = ir([("y", "x+5")], 20)
    original = updated.verify_ir_entry(example)
    override = updated.verify_ir_entry(example, override_claimed_val=15)
    # Use an excluded ID and an override ID to exercise the JSONL pipeline.
    excluded_id = sorted(excluded)[0]
    override_id = sorted(refs)[0]
    probe = work / "adjudication_probe.jsonl"
    reference = refs[override_id]
    probe_ir = ir([("y", "r")], 0, variables={"r": reference})
    probe_ir["claimed_answer"]["predicted_value"] = str(updated.clean_numeric_value(reference) + 1)
    rows = [{"case_id": excluded_id, "ir": example},
            {"case_id": override_id, "ir": probe_ir}]
    probe.write_text("".join(json.dumps(row) + "\n" for row in rows), encoding="utf-8")
    output_file = work / "adjudication_probe_results.json"
    updated.run_on_jsonl(probe, output_file, use_adjudicated_refs=True,
                        exclude_adjudicated=True, adjudication_csv=csv_path)
    out = json.loads(output_file.read_text())
    report["adjudication"] = {
        "excluded_count": len(excluded),
        "reference_override_count": len(refs),
        "counts_match_policy": len(excluded) == 6 and len(refs) == 24,
        "disjoint_ids": not (excluded & set(refs)),
        "original_claim": original,
        "reference_override": override,
        "claim_override_behaves_as_expected": original["status"] == "FAIL" and override["status"] == "PASS",
        "exclusion_applied": out["config"]["n_excluded_seen"] == 1 and
                             [r["case_id"] for r in out["results"]] == [override_id],
        "pipeline_override_applied": len(out["results"]) == 1 and
            out["results"][0]["verification"]["status"] == "PASS" and
            out["results"][0]["verification"]["claimed_value"] == reference and
            updated.verify_ir_entry(probe_ir)["status"] == "FAIL",
        "pipeline_output": out,
    }
except Exception as e:
    report["adjudication"] = {"error": repr(e)}
print("Adjudication:", {k: v for k, v in report["adjudication"].items() if k != "pipeline_output"})


## 6. Export results

Save the regression results, pilot comparisons, configuration and validation checks in a JSON report.

In [ ]:
aggregation_keys = ("matches_individual", "status_counts_match", "status_ids_match",
                    "record_count_matches", "total_count_matches")
adjudication_keys = ("counts_match_policy", "disjoint_ids", "claim_override_behaves_as_expected", "exclusion_applied", "pipeline_override_applied")
summary = {
    "synthetic_cases": len(results),
    "synthetic_matches": sum(r["match"] for r in results),
    "pilot_records": sum(len(rows) for rows in report["pilot_comparison"].values()),
    "pilot_exceptions": sum("error" in r for rows in report["pilot_comparison"].values() for r in rows),
    "unsat_core_check_passed": feedback["passed"],
    "aggregation_checks_passed": all(report["aggregation"].get(k) is True for k in aggregation_keys),
    "adjudication_checks_passed": all(report["adjudication"].get(k) is True for k in adjudication_keys),
}
report["summary"] = summary
print(json.dumps(summary, indent=2))
path = Path("FinVerify_Z3_Regression_Results.json")
path.write_text(json.dumps(report, indent=2), encoding="utf-8")
files.download(str(path))
